**Brief logic:** Introduces the notebook goal: build a RAG workflow using Azure AI Foundry Agents and Azure AI Search.

## RAG with AI Search and Foundry Agents


**Brief logic:** Shows the architecture flow from user question to agent, Azure AI Search retrieval, model grounding, and final answer.

![lab_flow](./rag-withAI-search.jpg)


**Brief logic:** Explains that the next cell installs the Python packages required for the RAG notebook.

### Installing Required Libraries


In [43]:
# Brief logic: Install the exact Python packages required to run this RAG notebook.
# azure-ai-projects is used for Foundry project/agent APIs.
# openai is used for conversation and response calls.
# python-dotenv loads settings from .env.
# azure-identity authenticates with Azure.
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Brief logic:** Explains that configuration values are loaded from `.env` instead of being hardcoded in the notebook.

### Setting Up Environment Variables


In [44]:
# Brief logic: Import required modules, load .env values, and store configuration variables for later cells.
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchAgentTool,
    PromptAgentDefinition,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

# Load variables from the local .env file into the Python process.
load_dotenv()

# Read Foundry and Azure AI Search settings that will be reused below.
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
ai_search_connection_name = os.getenv("AI_SEARCH_CONNECTION_NAME")
ai_search_index_name = os.getenv("AI_SEARCH_INDEX_NAME")


**Brief logic:** Explains that the next cell creates the main client for connecting to the Azure AI Foundry project.

### Setting up the Foundry Project Client


In [45]:
# Brief logic: Create a Foundry project client using the configured project endpoint and Azure credentials.
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)


**Brief logic:** Explains that the next cell creates an OpenAI-compatible client for conversations and responses.

### Creating the OpenAI Client


In [46]:
# Brief logic: Create the OpenAI-compatible client scoped to this Foundry project.
openai_client = project_client.get_openai_client()


**Brief logic:** Explains that the Azure AI Search connection ID must be resolved before it can be attached to the agent.

### Fetching the AI Search Connection ID


In [47]:
# Brief logic: Find the Azure AI Search connection by name and save its connection ID.
connection_id = ""

# Loop through all project connections until the configured search connection is found.
for connection in project_client.connections.list():
    if connection.name == ai_search_connection_name:
        connection_id = connection.id
        break

# Print the resolved connection ID for confirmation and troubleshooting.
print(f"The AI Search Connection ID is: {connection_id}")


The AI Search Connection ID is: /subscriptions/b0083ffe-3096-428e-9565-ebc0da7cce88/resourceGroups/rg-avyuktitraining1-8589/providers/Microsoft.CognitiveServices/accounts/ajay-agent-project111-resource/projects/ajay-agent-project111/connections/ajaysearchservice222xebi7o


**Brief logic:** Explains that the next cell creates the RAG agent and attaches Azure AI Search as the retrieval tool.

### Creating the RAG Agent


In [48]:
# Brief logic: Create a RAG agent version with model instructions and Azure AI Search retrieval configured.
agent = project_client.agents.create_version(
    agent_name="RAG-Agent",
    definition=PromptAgentDefinition(
        # Use the configured model deployment to generate answers.
        model=model_deployment_name,
        # Tell the agent to use AI Search for grounded RAG-style responses.
        instructions = "You are a helpful assistant that uses AI Search to answer user queries in a RAG setup",
        tools = [
            # Attach Azure AI Search as a tool available to this agent.
            AzureAISearchAgentTool(
                azure_ai_search = AzureAISearchToolResource(
                    indexes = [
                        AISearchIndexResource(
                            # Use the project connection ID resolved in the previous cell.
                            project_connection_id = connection_id,
                            # Search this configured Azure AI Search index.
                            index_name = ai_search_index_name,
                            # Use semantic retrieval for this pre-vectorized index.
                            # VECTOR_* query types require an integrated vectorizer on the Azure AI Search index.
                            query_type = AzureAISearchQueryType.SEMANTIC,
                            # Return the top 3 matching chunks/documents as grounding context.
                            top_k = 3
                        )
                    ]
                )
            )
        ]
    )
)

# Print the created agent details to confirm the agent name, version, and ID.
print(f"Created Agent: {agent.name} with version: {agent.version} and ID: {agent.id}")


Created Agent: RAG-Agent with version: 7 and ID: RAG-Agent:7


**Brief logic:** Explains that a conversation object stores the chat session used by the agent response call.

### Creating a Conversation Object for the Agent Chat System


In [49]:
# Brief logic: Create a new conversation object for this agent chat session.
conversation = openai_client.conversations.create()

# Print the conversation ID because the response call needs it.
print(f"Created conversation with id: {conversation.id}")


Created conversation with id: conv_20c85c534149e41a00JmdrQdRZNxUVS7yyhJSoYfGzcivwyU2l


**Brief logic:** Explains that the next cells define the user query and send it to the RAG agent for a grounded answer.

### Calling the Agent and Streaming the Response


In [50]:
# Brief logic: Define the user question that will be sent to the RAG agent.
user_input = "What are the payroll policies present in HR POLICY Docs for ABC Corp?"

# Optional test queries. Uncomment one to test the same RAG flow with a different question.
# user_input = "What are the payroll policies present in HR POLICY Docs for ABC Corp?."
# user_input = "Tell me about the employee records in the ABC Corp dataset."


In [51]:
# Brief logic: Send the question to the agent, require search tool usage, and print the final grounded answer.
response = openai_client.responses.create(
        tool_choice="required",
        conversation=conversation.id,
        input=user_input,
        # Reference the Foundry agent created earlier instead of calling the model directly.
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

# Display the final response generated after retrieval and grounding.
print(f"Agent Response: {response.output_text}")


Agent Response: The payroll policies outlined in ABC Corp's HR Policy documents include the following details:

1. **Salary Processing**: Salaries are credited monthly to employees' accounts, with deductions made as per statutory norms.
2. **Payslips Access**: Employees can view their payslips via the employee portal【4:0†source】【4:1†source】.
3. **Employee Record Validation**: Employee payroll eligibility is validated using records stored in the HR-Employee_Records dataset. Access to these records is restricted to authorized HR and payroll users【4:0†source】.
4. **General Information Access**: While employees can access policy summaries, they cannot view individual payroll records【4:0†source】.

If specific queries about payroll processes or exceptions arise, further details can be drawn from the described exceptions handling in the documents.
